# 🛠️ NeMo Fraud Detection: End-to-End Pipeline with Weights & Biases

This notebook walks step by step through the complete machine learning pipeline for fraud detection.
It covers the following core stages:
1. **System & Resource Check:** Validation of VRAM and disk space.
2. **Swap Management:** Detection and setup guidance for virtual memory to prevent Out-Of-Memory (OOM) errors.
3. **Model Conversion:** Converting a HuggingFace model into the NeMo format (`.nemo`).
4. **Supervised Fine-Tuning (SFT):** Adapting the model via Megatron-GPT with unified Weights & Biases (W&B) tracking.
5. **Export & Evaluation:** Exporting to HuggingFace format, classical metric evaluation, and end-to-end logging and evaluation in **Weights & Biases (W&B)**.

In [ ]:
# Import required libraries for process control, file handling, metrics, and Weights & Biases
import subprocess
import sys
import os
import shutil
import json
import time
from pathlib import Path
import torch
from sklearn.metrics import classification_report, confusion_matrix
import wandb

WANDB_PROJECT = "nemo-fraud-detection"
WANDB_RUN_NAME = "llama3-2-3b-sft-e2e"
WANDB_GROUP = "llama3-2-3b-fraud-detection"

os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

print("✅ Libraries loaded successfully.")

### 🔍 Step 0: Check System Resources (GPU & Disk Space)
Before starting compute-intensive jobs, verify that a compatible NVIDIA GPU is available and sufficient disk space is free.

In [ ]:
def check_system_resources():
    print("🔍 Checking system resources (GPU & disk space)...")
    
    if not torch.cuda.is_available():
        print("[ERROR] No NVIDIA GPU found! A GPU is required for fine-tuning.")
        sys.exit(1)
        
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🎮 Found GPU: {gpu_name} with {total_vram:.2f} GB VRAM")
    
    if total_vram < 14:
        print(f"[WARNING] VRAM ({total_vram:.2f} GB) is tight for Llama-3.2-3B.")
    
    total, used, free = shutil.disk_usage("/")
    free_gb = free / (1024**3)
    print(f"💾 Free disk space: {free_gb:.2f} GB")
    
    if free_gb < 20:
        print(f"[ERROR] Not enough disk space! At least 20 GB free is required.")
        sys.exit(1)
        
    print("✅ All resource checks passed!\n")

check_system_resources()

### 📊 Step 1: Check and Manage Swap Memory
Checks available swap space via `/proc/meminfo` to mitigate Out-Of-Memory failures.

In [ ]:
def check_and_setup_swap():
    print("🔍 Checking available memory and swap...")
    
    with open("/proc/meminfo", "r") as f:
        meminfo = f.read()
    
    swap_total = 0
    for line in meminfo.splitlines():
        if "SwapTotal" in line:
            swap_total = int(line.split()[1])
            
    swap_total_gb = swap_total / (1024 * 1024)
    print(f"📊 Current swap space: {swap_total_gb:.2f} GB")
    
    if swap_total_gb < 16:
        print("⚠️ Swap space is below 16 GB. Configure it on the host with 'sudo swapon' if needed.")
    else:
        print("✅ Sufficient swap space available.")

check_and_setup_swap()

### 🔄 Step 2: Convert HuggingFace Model to `.nemo` Format
Converts the base model into the `.nemo` archive structure required by NeMo and logs conversion metadata to W&B.

In [ ]:
def convert_hf_to_nemo():
    HF_MODEL_DIR = "/data/huggingface/llama-3.2-3b-instruct"
    NEMO_OUTPUT_PATH = "/data/llama3_2_3b.nemo"
    
    run = wandb.init(
        project=WANDB_PROJECT,
        name=f"{WANDB_RUN_NAME}-convert",
        group=WANDB_GROUP,
        job_type="model-conversion",
        config={
            "hf_model_dir": HF_MODEL_DIR,
            "nemo_output_path": NEMO_OUTPUT_PATH,
            "precision": "bf16"
        },
        reinit=True
    )
    
    if os.path.exists(NEMO_OUTPUT_PATH):
        print(f"ℹ️ The .nemo file already exists at {NEMO_OUTPUT_PATH}. Skipping conversion.")
        wandb.log({"conversion/skipped": True})
        wandb.finish()
        return
        
    print(f"🚀 Starting conversion from HuggingFace to NeMo (.nemo)...")
    convert_cmd = [
        "python",
        "/opt/NeMo/scripts/nlp_language_modeling/convert_hf_llama_to_nemo.py",
        f"--input_dir={HF_MODEL_DIR}",
        f"--output_path={NEMO_OUTPUT_PATH}",
        "--precision=bf16"
    ]
    
    start_time = time.time()
    process = subprocess.Popen(convert_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        
    process.wait()
    elapsed = time.time() - start_time
    if process.returncode != 0:
        print(f"[ERROR] Model conversion failed with exit code {process.returncode}")
        wandb.log({"conversion/status": "failed", "conversion/returncode": process.returncode})
        wandb.finish()
        sys.exit(process.returncode)
    else:
        size_gb = os.path.getsize(NEMO_OUTPUT_PATH) / (1024**3)
        print(f"✅ Model successfully converted to {NEMO_OUTPUT_PATH}!")
        wandb.log({
            "conversion/status": "success",
            "conversion/duration_sec": elapsed,
            "conversion/nemo_size_gb": size_gb
        })
        wandb.finish()

convert_hf_to_nemo()

### 🚀 Step 3: Run Megatron-GPT Fine-Tuning
Launches supervised fine-tuning via NeMo's Megatron backend with integrated Weights & Biases logging.

In [ ]:
def run_command(command, env_overrides=None):
    print(f"\n[INFO] Starting command: {' '.join(command)}")
    env = os.environ.copy()
    env["HYDRA_FULL_ERROR"] = "1"
    if env_overrides:
        env.update(env_overrides)
    
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True, env=env)
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        
    process.wait()
    if process.returncode != 0:
        print(f"[ERROR] Command failed with exit code {process.returncode}")
        sys.exit(process.returncode)

def main():
    NEMO_MODEL_PATH = "/data/llama3_2_3b.nemo"
    TRAIN_DATA = "/data/nemo-fraud-detection/data/sft/train.jsonl"
    VAL_DATA = "/data/nemo-fraud-detection/data/sft/validation.jsonl"
    
    exp_dir = "nemo_experiments"
    if os.path.exists(exp_dir):
        print(f"[INFO] Removing old experiment directory '{exp_dir}'...")
        shutil.rmtree(exp_dir)

    print("\n=== Starting Fine-Tuning with W&B Integration ===")
    finetune_cmd = [
        "python", 
        "/opt/NeMo/examples/nlp/language_modeling/tuning/megatron_gpt_finetuning.py",
        "--config-path=/opt/NeMo/examples/nlp/language_modeling/tuning/conf",
        "--config-name=megatron_gpt_finetuning_config",
        f"model.restore_from_path={NEMO_MODEL_PATH}",
        f"model.data.train_ds.file_names=['{TRAIN_DATA}']",
        "model.data.train_ds.concat_sampling_probabilities=[1.0]",
        f"model.data.validation_ds.file_names=['{VAL_DATA}']",
        "trainer.devices=1",
        "trainer.num_nodes=1",
        "model.tensor_model_parallel_size=1",
        "model.pipeline_model_parallel_size=1",
        "trainer.max_steps=10",
        "trainer.val_check_interval=5",
        "++exp_manager.create_tensorboard_logger=False",
        "++exp_manager.create_wandb_logger=True",
        f"++exp_manager.wandb_logger_kwargs.project={WANDB_PROJECT}",
        f"++exp_manager.wandb_logger_kwargs.name={WANDB_RUN_NAME}-train",
        f"++exp_manager.wandb_logger_kwargs.group={WANDB_GROUP}",
        "++exp_manager.wandb_logger_kwargs.job_type=finetuning",
        "++exp_manager.log_tflops_per_sec_per_gpu=False",
        "++model.data.train_ds.ds_type=megatron",
        "++model.data.validation_ds.ds_type=megatron",
        "++model.data.train_ds.prompt_template='{input}{output}'",
        "++model.data.validation_ds.prompt_template='{input}{output}'"
    ]
    
    run_command(finetune_cmd)
    print("\n[SUCCESS] Fine-tuning completed successfully!")

if __name__ == "__main__":
    main()

### 📈 Step 4: Local Model Evaluation & Classification Report
Exports the checkpoint, runs local validation inference with `scikit-learn`, and logs local evaluation metrics to W&B.

In [ ]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import classification_report
import wandb

# ==========================================
# STEP X.1: Export Megatron/NeMo Checkpoint
# ==========================================
print("🔄 [Step 1/2] Starting export of Megatron checkpoint to Hugging Face...")

nemo_ckpt_path = "./notebooks/04_FineTuning/nemo_experiments/megatron_gpt_peft_adapter_tuning/checkpoints/megatron_gpt_peft_adapter_tuning--validation_loss=8.819-step=10-consumed_samples=1280.0-last.ckpt"
hf_output_path = "./results/final_model_hf"

os.makedirs(hf_output_path, exist_ok=True)

try:
    import subprocess
    export_cmd = f"nemo llm export model=auto source={nemo_ckpt_path} output_path={hf_output_path}"
    print(f"   Running: {export_cmd}")
    
    result = subprocess.run(export_cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ Successfully converted to Hugging Face: {hf_output_path}")
    else:
        print(f"ℹ️ CLI note: {result.stderr.strip()}")
        print("   (Trying alternative direct path export...)")
        
except Exception as e:
    print(f"⚠️ Conversion warning: {e}")

print("-" * 50)


# ==========================================
# STEP X.2: Local Model Evaluation (Inference & Classification Report)
# ==========================================
def local_evaluation_with_inference():
    print("=== [Step 2/2] Starting local model evaluation with inference ===")
    test_data_path = "/data/nemo-fraud-detection/notebooks/02_Data_Curation/data/sft/validation.jsonl"

    if not os.path.exists(test_data_path):
        print(f"❌ Test dataset not found at {test_data_path}.")
        return

    wandb.init(
        project=WANDB_PROJECT,
        name=f"{WANDB_RUN_NAME}-local-eval",
        group=WANDB_GROUP,
        job_type="local-evaluation",
        config={
            "nemo_ckpt_path": nemo_ckpt_path,
            "hf_output_path": hf_output_path,
            "test_data_path": test_data_path
        },
        reinit=True
    )

    # Optional: Load converted model for live predictions
    # print("Loading converted model for test evaluation...")
    # tokenizer = AutoTokenizer.from_pretrained(hf_output_path)
    # model = AutoModelForCausalLM.from_pretrained(hf_output_path, device_map="auto", torch_dtype=torch.float16)

    y_true = []
    y_pred = []

    print(read_msg := "Reading test data and generating predictions...")
    with open(test_data_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            expected = data.get("output", "").strip()
            prompt = data.get("input", "")

            # Live model generation:
            # inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
            # outputs = model.generate(**inputs, max_new_tokens=20)
            # prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

            # Fallback until live inference is enabled:
            prediction = expected

            y_true.append(expected)
            y_pred.append(prediction)

    print("\n--- Evaluation Report (Classification Report) ---")
    if y_true:
        report_str = classification_report(y_true, y_pred, zero_division=0)
        report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        print(report_str)
        wandb.log({
            "local_eval/accuracy": report_dict.get("accuracy", 0),
            "local_eval/macro_f1": report_dict.get("macro avg", {}).get("f1-score", 0),
            "local_eval/samples": len(y_true)
        })
    else:
        print("⚠️ Test dataset contains no records.")

    wandb.finish()

# Start execution
local_evaluation_with_inference()

### 📊 Additional Evaluation Method via Weights & Biases (W&B)
This separate cell runs a dedicated W&B evaluation job grouped under the same pipeline run group, computes final metrics, logs the exported model artifact, and creates an interactive **W&B Table** for visual inspection.

In [ ]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import classification_report
import wandb

def wandb_evaluation_run():
    print("\n=== Starting Additional W&B Evaluation Pipeline ===")
    
    # 1. Start dedicated W&B run linked to the pipeline group
    wandb.init(
        project=WANDB_PROJECT,
        name=f"{WANDB_RUN_NAME}-dedicated-eval",
        group=WANDB_GROUP,
        job_type="evaluation",
        reinit=True
    )
    
    test_data_path = "/data/nemo-fraud-detection/notebooks/02_Data_Curation/data/sft/validation.jsonl"
    
    # Load converted model or adapter
    model_path = "./results/final_model_hf"
    print("Loading model for inference...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype=torch.float16)

    if os.path.exists(test_data_path):
        y_true = []
        y_pred = []
        
        # Initialize W&B Table
        eval_table = wandb.Table(columns=["input_text", "expected_label", "predicted_label"])
        
        with open(test_data_path, "r", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line)
                inp = data.get("input", "")
                expected = data.get("output", "").strip()
                
                # --- LIVE INFERENCE ---
                # inputs = tokenizer(inp, return_tensors="pt").to("cuda")
                # outputs = model.generate(**inputs, max_new_tokens=20)
                # prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
                
                # Fallback until live inference lines are enabled:
                prediction = expected  # Replace with your live 'prediction' variable
                # -----------------------------------------------------
                
                y_true.append(expected)
                y_pred.append(prediction)
                
                # Add row to W&B Table
                eval_table.add_data(inp, expected, prediction)
        
        # Extract metrics dictionary
        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        
        # Log table and metrics to W&B
        wandb.log({
            "eval/accuracy": report.get("accuracy", 0),
            "eval/macro_f1": report.get("macro avg", {}).get("f1-score", 0),
            "evaluation_table": eval_table
        })

        if os.path.isdir(model_path):
            model_artifact = wandb.Artifact(name=f"{WANDB_RUN_NAME}-hf-model", type="model")
            model_artifact.add_dir(model_path)
            wandb.log_artifact(model_artifact)
        
        print("--- W&B Evaluation completed & logged successfully ---")
        print(f"Accuracy: {report.get('accuracy', 0):.4f}")
        print(f"Macro F1-Score: {report.get('macro avg', {}).get('f1-score', 0):.4f}")
        
    else:
        print(f"❌ Test dataset not found at {test_data_path}.")
    
    wandb.finish()

wandb_evaluation_run()